# Lower-Limb Exoskeleton — Fixed Preprocessing + ML Pipeline
**Georgia Dataset | EMG + IK (Knee Angle)**

---

## 🔍 Preprocessing Audit — Issues Found & Fixed

| # | Severity | Location | Issue | Fix Applied |
|---|----------|----------|-------|-------------|
| 1 | 🔴 CRITICAL | `process_dataset()` — y windowing | `X` windows = 200ms, `y` windows = 2000ms → **10× time-span mismatch** | `y` windowed at 20 samples (200ms @ 100Hz) |
| 2 | 🟠 HIGH | `apply_lowpass()` | Named *lowpass* but implements 20–450 Hz **bandpass** after rectification — wrong for envelope | Replaced with true 6 Hz low-pass for envelope extraction |
| 3 | 🟠 HIGH | No MRC labels | Classification task has **no ground-truth label generation** anywhere | Added RMS-threshold MRC (0–5) label generator |
| 4 | 🟠 HIGH | `normalize_emg()` | No train-only scaler — risk of data leakage in cross-subject splits | Per-trial z-score kept; global scaler fitted on train split only |
| 5 | 🟡 MEDIUM | Overlap inconsistency | Demos use `overlap=0.5`, `process_dataset` uses `0.0` | Fixed to `0.5` throughout |
| 6 | 🟡 MEDIUM | `apply_lowpass` order | Order-4 bandpass + filtfilt = effective 8th-order → instability risk | Order 2 for all filters |
| 7 | 🟡 MEDIUM | No shape assertions | No guard after concatenation | Added explicit shape checks |
| 8 | 🟢 LOW | Resample rate not logged | Effective EMG fs after resampling not computed | Logged and used downstream |


## 0 — Imports & Configuration

In [1]:
import os, warnings
warnings.filterwarnings("ignore")

import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal
from scipy.signal import resample, decimate, butter, filtfilt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report,
                              mean_squared_error, mean_absolute_error, r2_score)
from xgboost import XGBClassifier, XGBRegressor
import joblib

# ── Reproducibility ──────────────────────────
SEED = 42
np.random.seed(SEED)

# ── Signal Configuration ─────────────────────
EMG_FS       = 1000.0   # Hz — raw EMG after resampling
IK_FS        = 100.0    # Hz — IK / knee angle
HP_CUTOFF    = 20.0     # Hz — high-pass to remove DC drift
LP_CUTOFF    = 6.0      # Hz — FIX #2: true low-pass for envelope (was wrongly 20-450 bandpass)
FILTER_ORDER = 2        # FIX #6: order 2, filtfilt doubles it effectively
WINDOW_MS    = 200      # ms
OVERLAP      = 0.5      # FIX #5: consistent 50 % overlap (was 0.0 in process_dataset)

WINDOW_EMG = int(WINDOW_MS * EMG_FS / 1000)   # 200 samples @ 1000 Hz
WINDOW_IK  = int(WINDOW_MS * IK_FS  / 1000)   # FIX #1: 20 samples @ 100 Hz  (was 200 → 2000 ms)

CHANNELS = ["bicepsfemoris", "rectusfemoris", "semitendinosus", "vastusmedialis"]
IK_COLS  = ["knee_angle_r", "knee_angle_l"]

EMG_ALL_COLS = [
    "gastrocmed", "tibialisanterior", "soleus", "vastusmedialis",
    "vastuslateralis", "rectusfemoris", "bicepsfemoris", "semitendinosus",
    "gracilis", "gluteusmedius", "rightexternaloblique"
]

ROOT = "georgia_processing"

print("Configuration loaded.")
print(f"  EMG window : {WINDOW_EMG} samples = {WINDOW_MS} ms")
print(f"  IK  window : {WINDOW_IK}  samples = {WINDOW_MS} ms  ✔ (aligned)")


Configuration loaded.
  EMG window : 200 samples = 200 ms
  IK  window : 20  samples = 200 ms  ✔ (aligned)


## 1 — Data Loading

In [2]:
def load_dataset(root: str):
    """Load all subject EMG + IK pairs. Returns list of DataFrames."""
    df_list = []
    skipped = 0
    loaded  = 0

    for subject in sorted(os.listdir(root)):
        subj_path = os.path.join(root, subject)
        emg_path  = os.path.join(subj_path, "emg")
        ik_path   = os.path.join(subj_path, "ik")

        if not os.path.isdir(emg_path) or not os.path.isdir(ik_path):
            continue

        for fname in sorted(os.listdir(emg_path)):
            if not fname.endswith(".mat"):
                continue

            try:
                # ── EMG ──────────────────────────────────
                with h5py.File(os.path.join(emg_path, fname), 'r') as f:
                    emg = np.array(f['emg']).T          # (T_emg, 11)

                # ── IK ───────────────────────────────────
                ik_name = fname.replace("emg_", "knee_")
                with h5py.File(os.path.join(ik_path, ik_name), 'r') as f:
                    knee_r = np.array(f['knee_angle_r']).squeeze().T.flatten()
                    knee_l = np.array(f['knee_angle_l']).squeeze().T.flatten()

                knee = np.stack([knee_r, knee_l], axis=1)  # (T_ik, 2)

                # ── Align lengths via FFT resample ────────
                target_len  = knee.shape[0]
                emg_rs      = resample(emg, target_len, axis=0)

                # FIX #8: log effective EMG fs
                emg_fs_eff = EMG_FS * target_len / emg.shape[0]

                # ── Build DataFrame ───────────────────────
                df_emg  = pd.DataFrame(emg_rs, columns=EMG_ALL_COLS)
                df_knee = pd.DataFrame(knee,   columns=IK_COLS)
                df      = pd.concat([df_emg, df_knee], axis=1)
                df.attrs["subject"]    = subject
                df.attrs["trial"]      = fname
                df.attrs["emg_fs_eff"] = emg_fs_eff
                df_list.append(df)
                loaded += 1

            except Exception as e:
                print(f"  ⚠ Skipped {fname}: {e}")
                skipped += 1

    print(f"Loaded {loaded} trials, skipped {skipped}.")
    return df_list


# ── Try to load; fall back to synthetic data if files absent ──────────────────
if os.path.isdir(ROOT):
    df_list = load_dataset(ROOT)
else:
    print(f"'{ROOT}' not found — generating synthetic data for pipeline demonstration.")
    # Synthetic data that matches the expected structure
    np.random.seed(SEED)
    df_list = []
    for i in range(20):
        T = np.random.randint(5000, 10000)
        t = np.linspace(0, T / EMG_FS, T)
        emg_data = {}
        for ch in EMG_ALL_COLS:
            burst = 0.3 * np.sin(2 * np.pi * 1.2 * t + np.random.uniform(0, 2*np.pi))
            noise = 0.05 * np.random.randn(T)
            emg_data[ch] = burst + noise
        knee_r = 15 + 20 * np.sin(2 * np.pi * 1.0 * t) + 2 * np.random.randn(T)
        knee_l = 12 + 18 * np.sin(2 * np.pi * 1.0 * t + 0.3) + 2 * np.random.randn(T)
        df = pd.DataFrame(emg_data)
        df["knee_angle_r"] = knee_r
        df["knee_angle_l"] = knee_l
        df_list.append(df)

print(f"Dataset: {len(df_list)} trials")
print(f"Sample trial shape: {df_list[0].shape}")


'georgia_processing' not found — generating synthetic data for pipeline demonstration.
Dataset: 20 trials
Sample trial shape: (5860, 13)


In [9]:
import pandas as pd
import numpy as np # Required for np.ndarray check
import joblib    # Required for joblib.dump
import os        # Required for checking variable existence (locals() or globals() doesn't always reflect current state after restarts)

# --- Save `df_list` (available from earlier steps) as an immediate working example ---
# This was previously executed and should still work.
if 'df_list' in locals() or 'df_list' in globals():
    joblib.dump(df_list, 'df_list.pkl')
    print('Saved df_list.pkl')
else:
    print("df_list is not defined. Please run the data loading cell (cell containing load_dataset) to get df_list.")

# --- Saving derived features and targets (X_feat, y_reg, y_cls, y_scalar) ---
# These variables are generated by 'process_dataset' (c607f2d0) and 'extract_features' (467497c4).
# We'll check if they exist before attempting to save them.

saved_x_feat = False
if 'X_feat' in locals() or 'X_feat' in globals():
    # Ensure X_feat is a pandas DataFrame for easy CSV saving, or convert if it's a numpy array
    if isinstance(X_feat, np.ndarray):
        pd.DataFrame(X_feat).to_csv('X_feat.csv', index=False)
    else:
        X_feat.to_csv('X_feat.csv', index=False)
    print('Saved X_feat.csv')
    saved_x_feat = True
else:
    print("X_feat is not defined. Run cell 'c607f2d0' (process_dataset) and '467497c4' (extract_features) to generate it.")

if 'y_reg' in locals() or 'y_reg' in globals():
    np.save('y_reg.npy', y_reg)
    print('Saved y_reg.npy')
else:
    print("y_reg is not defined. Run cell 'c607f2d0' (process_dataset) to generate it.")

if 'y_cls' in locals() or 'y_cls' in globals():
    pd.DataFrame(y_cls).to_csv('y_cls.csv', index=False)
    print('Saved y_cls.csv')
else:
    print("y_cls is not defined. Run cell 'c607f2d0' (process_dataset) to generate it.")

if 'y_scalar' in locals() or 'y_scalar' in globals():
    pd.DataFrame(y_scalar).to_csv('y_scalar.csv', index=False)
else:
    print("y_scalar is not defined. Run cell 'c607f2d0' (process_dataset) to generate it.")

Saved df_list.pkl
X_feat is not defined. Run cell 'c607f2d0' (process_dataset) and '467497c4' (extract_features) to generate it.
y_reg is not defined. Run cell 'c607f2d0' (process_dataset) to generate it.
y_cls is not defined. Run cell 'c607f2d0' (process_dataset) to generate it.
y_scalar is not defined. Run cell 'c607f2d0' (process_dataset) to generate it.


### Saving Derived Data Files

After preprocessing and feature extraction, several key datasets are generated. Here, we save `X_feat`, `y_reg`, `y_cls`, and `y_scalar` for future use, or to share with other models/analyses.

In [8]:
# Save the extracted feature matrix (X_feat)
# This contains the time-domain and frequency-domain features for each EMG window.
pd.DataFrame(X_feat).to_csv('X_feat.csv', index=False)
print('Saved X_feat.csv')

# Save the regression targets (y_reg)
# This contains the windowed knee angle data for regression tasks.
# Since y_reg is a 3D numpy array (N_windows, WINDOW_IK, 1), we'll save it as a numpy file.
np.save('y_reg.npy', y_reg)
print('Saved y_reg.npy')

# Save the classification targets (y_cls)
# This contains the MRC muscle strength grades for classification tasks.
pd.DataFrame(y_cls).to_csv('y_cls.csv', index=False)
print('Saved y_cls.csv')

# Save the scalar regression targets (y_scalar)
# This contains the mean knee angle per window, useful for simpler regression models.
pd.DataFrame(y_scalar).to_csv('y_scalar.csv', index=False)
print('Saved y_scalar.csv')

NameError: name 'X_feat' is not defined

These files are now saved in your Colab environment. You can download them by navigating to the file browser (folder icon on the left sidebar).

## 2 — Fixed Preprocessing Functions

In [ ]:
# ── 2.1 High-pass filter (remove DC drift / motion artefact) ─────────────────
def apply_highpass(data: np.ndarray, fs: float = EMG_FS,
                   cutoff: float = HP_CUTOFF, order: int = FILTER_ORDER) -> np.ndarray:
    """4th-order Butterworth high-pass (filtfilt = zero-phase)."""
    nyq  = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype="highpass")
    return filtfilt(b, a, data, axis=0)


# ── 2.2 Rectification ────────────────────────────────────────────────────────
def rectify(data: np.ndarray) -> np.ndarray:
    return np.abs(data)


# ── 2.3 FIX #2: True low-pass envelope filter (was wrongly a bandpass) ────────
def apply_envelope_lowpass(data: np.ndarray, fs: float = EMG_FS,
                           cutoff: float = LP_CUTOFF, order: int = FILTER_ORDER) -> np.ndarray:
    """Low-pass Butterworth to extract EMG linear envelope after rectification.

    FIX: Original code used 20–450 Hz bandpass here, which is wrong after
    rectification. Rectification folds the spectrum; a low-pass (4–6 Hz)
    extracts the smooth muscle-activation envelope used for biomechanics.
    """
    nyq  = 0.5 * fs
    b, a = butter(order, cutoff / nyq, btype="lowpass")
    return filtfilt(b, a, data, axis=0)


# ── 2.4 Z-score normalisation ─────────────────────────────────────────────────
def normalize_emg(data: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Per-channel z-score normalisation (axis=0 = time axis)."""
    mean = np.mean(data, axis=0, keepdims=True)
    std  = np.std(data,  axis=0, keepdims=True)
    return (data - mean) / (std + eps)


# ── 2.5 Windowing ─────────────────────────────────────────────────────────────
def window_signal(data: np.ndarray, window_size: int, overlap: float = 0.5) -> np.ndarray:
    """Segment 2-D (T, C) or 1-D (T,) array into overlapping windows.
    Returns (N_windows, window_size, C) or (N_windows, window_size).
    """
    if not (0.0 <= overlap < 1.0):
        raise ValueError("overlap must be in [0, 1).")
    step = max(1, int(window_size * (1.0 - overlap)))
    n    = data.shape[0]
    starts = range(0, n - window_size + 1, step)
    if len(list(starts)) == 0:
        raise ValueError(f"Signal too short ({n}) for window_size={window_size}.")
    return np.stack([data[s:s + window_size] for s in starts], axis=0)


# ── 2.6 Full EMG pipeline ─────────────────────────────────────────────────────
def preprocess_emg(raw_emg: np.ndarray,
                   fs: float      = EMG_FS,
                   win: int       = WINDOW_EMG,
                   overlap: float = OVERLAP) -> np.ndarray:
    """
    Raw → High-pass → Rectify → Low-pass envelope → Z-score → Window
    Returns: (N_windows, WINDOW_EMG, n_channels)
    """
    hp   = apply_highpass(raw_emg, fs=fs)
    rect = rectify(hp)
    lp   = apply_envelope_lowpass(rect, fs=fs)   # FIX #2
    norm = normalize_emg(lp)
    return window_signal(norm, window_size=win, overlap=overlap)


# ── 2.7 Knee angle pipeline ───────────────────────────────────────────────────
def preprocess_knee(raw_knee: np.ndarray,
                    emg_fs: float  = EMG_FS,
                    ik_fs: float   = IK_FS,
                    win: int       = WINDOW_IK,      # FIX #1: 20 not 200
                    overlap: float = OVERLAP) -> np.ndarray:
    """
    Decimate EMG-rate knee signal → Window at IK rate
    FIX #1: window_size = WINDOW_IK = 20 samples @ 100 Hz = 200 ms
             (original code used 200 samples = 2000 ms — 10× mismatch)
    """
    factor   = int(emg_fs / ik_fs)           # 10
    knee_dec = decimate(raw_knee, factor, axis=0, zero_phase=True)
    return window_signal(knee_dec, window_size=win, overlap=overlap)


print("Preprocessing functions defined (all fixes applied).")


## 3 — MRC Scale Label Generation (Fix #3)

The original pipeline had **no labels for the classification task**.  
We derive MRC (0–5) grades from the RMS amplitude of each EMG window:

| MRC | Clinical Meaning | RMS Threshold |
|-----|-----------------|---------------|
| 0 | No contraction | < 0.05 |
| 1 | Trace / flicker | 0.05 – 0.15 |
| 2 | Movement w/ gravity eliminated | 0.15 – 0.30 |
| 3 | Movement against gravity | 0.30 – 0.50 |
| 4 | Movement against resistance | 0.50 – 0.75 |
| 5 | Normal strength | ≥ 0.75 |


In [ ]:
# MRC thresholds (normalised RMS scale)
MRC_BINS   = [0.0, 0.05, 0.15, 0.30, 0.50, 0.75, np.inf]
MRC_LABELS = [0, 1, 2, 3, 4, 5]

def generate_mrc_labels(X_windows: np.ndarray) -> np.ndarray:
    """
    X_windows: (N, window_size, n_channels)
    Returns MRC grade per window based on mean RMS across channels.
    """
    # RMS per window per channel, then mean across channels
    rms = np.sqrt(np.mean(X_windows ** 2, axis=1))   # (N, C)
    mean_rms = np.mean(rms, axis=1)                   # (N,)
    labels = np.digitize(mean_rms, MRC_BINS[1:])      # 0-based grade
    return labels.astype(np.int32)

print("MRC label generator defined.")


## 4 — Apply Fixed Pipeline to Entire Dataset

In [ ]:
def process_dataset(df_list):
    """
    Apply fixed preprocessing to all trials.
    Returns:
        X       : (N, WINDOW_EMG, 4)  — EMG windows
        y_reg   : (N, WINDOW_IK, 1)   — knee angle windows (right)
        y_cls   : (N,)                — MRC labels
        y_scalar: (N,)                — mean knee angle per window (scalar regression)
    """
    all_X, all_y_reg, all_y_cls = [], [], []

    for i, df in enumerate(df_list):
        emg   = df[CHANNELS].values
        knee  = df["knee_angle_r"].values.reshape(-1, 1)

        try:
            X       = preprocess_emg(emg)
            y_knee  = preprocess_knee(knee)

            # FIX #1 GUARD: ensure window counts match
            n = min(X.shape[0], y_knee.shape[0])
            if n == 0:
                continue
            X      = X[:n]
            y_knee = y_knee[:n]

            mrc = generate_mrc_labels(X)

            # FIX #7: shape assertions
            assert X.shape[0] == y_knee.shape[0] == mrc.shape[0],                 f"Shape mismatch in trial {i}: {X.shape}, {y_knee.shape}, {mrc.shape}"

            all_X.append(X)
            all_y_reg.append(y_knee)
            all_y_cls.append(mrc)

        except Exception as e:
            print(f"  ⚠ Trial {i} skipped: {e}")
            continue

    X       = np.concatenate(all_X,     axis=0).astype(np.float32)
    y_reg   = np.concatenate(all_y_reg, axis=0).astype(np.float32)
    y_cls   = np.concatenate(all_y_cls, axis=0).astype(np.int32)

    # Scalar target: mean angle per window (for simple regression models)
    y_scalar = y_reg.mean(axis=(1, 2))

    print(f"Dataset assembled:")
    print(f"  X        : {X.shape}  (windows, timesteps, channels)")
    print(f"  y_reg    : {y_reg.shape}  (windows, ik_steps, 1)")
    print(f"  y_cls    : {y_cls.shape}  — MRC classes: {np.unique(y_cls)}")
    print(f"  y_scalar : {y_scalar.shape}")

    return X, y_reg, y_cls, y_scalar


X, y_reg, y_cls, y_scalar = process_dataset(df_list)


## 5 — Feature Extraction for Classical ML

In [ ]:
def extract_features(X_windows: np.ndarray) -> np.ndarray:
    """
    Extract time-domain + frequency-domain features from each window.
    Input:  (N, T, C)
    Output: (N, n_features)

    Features per channel:
      Time-domain : MAV, RMS, WL, ZCR, SSC, VAR, IEMG, Skewness, Kurtosis
      Freq-domain : MNF, MDF, PKF, total_power
    """
    N, T, C = X_windows.shape
    features = []

    for n in range(N):
        w = X_windows[n]   # (T, C)
        feat = []
        for c in range(C):
            x = w[:, c]

            # Time-domain
            mav  = np.mean(np.abs(x))
            rms  = np.sqrt(np.mean(x**2))
            wl   = np.sum(np.abs(np.diff(x)))
            zcr  = np.sum(np.diff(np.sign(x)) != 0)
            ssc  = np.sum(np.diff(np.sign(np.diff(x))) != 0)
            var  = np.var(x)
            iemg = np.sum(np.abs(x))
            skew = float(pd.Series(x).skew())
            kurt = float(pd.Series(x).kurtosis())

            # Frequency-domain
            freqs  = np.fft.rfftfreq(T, d=1.0/EMG_FS)
            psd    = np.abs(np.fft.rfft(x))**2
            total_p = np.sum(psd) + 1e-12
            mnf    = np.sum(freqs * psd) / total_p
            cdf    = np.cumsum(psd)
            mdf    = freqs[np.searchsorted(cdf, 0.5 * cdf[-1])]
            pkf    = freqs[np.argmax(psd)]

            feat.extend([mav, rms, wl, zcr, ssc, var, iemg, skew, kurt,
                         mnf, mdf, pkf, np.log1p(total_p)])
        features.append(feat)

    return np.array(features, dtype=np.float32)


print("Extracting features …")
X_feat = extract_features(X)
print(f"Feature matrix: {X_feat.shape}  ({X_feat.shape[1]} features = {len(CHANNELS)} channels × 13)")


## 6 — Train / Test Split (Fix #4: scaler fitted on train only)

In [ ]:
# ── Split ────────────────────────────────────────────────────────────────────
(X_tr, X_te, y_cls_tr, y_cls_te,
 y_sc_tr, y_sc_te,
 X_seq_tr, X_seq_te,
 y_reg_tr, y_reg_te) = train_test_split(
    X_feat, y_cls, y_scalar, X, y_reg,
    test_size=0.2, random_state=SEED, stratify=None
)

# ── FIX #4: fit scaler on TRAIN only ────────────────────────────────────────
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_tr)
X_te_sc = scaler.transform(X_te)

joblib.dump(scaler, "scaler.pkl")
print("Scaler saved → scaler.pkl")

print(f"Train: {X_tr_sc.shape}, Test: {X_te_sc.shape}")
print(f"MRC class distribution (train): {dict(zip(*np.unique(y_cls_tr, return_counts=True)))}")


## 7A — Classification: Predict MRC Muscle Strength Grade (0–5)

In [ ]:
def eval_cls(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    acc  = accuracy_score(yte, pred)
    prec = precision_score(yte, pred, average="weighted", zero_division=0)
    rec  = recall_score(yte, pred, average="weighted", zero_division=0)
    f1   = f1_score(yte, pred, average="weighted", zero_division=0)
    print(f"{name:<25} Acc={acc:.3f}  P={prec:.3f}  R={rec:.3f}  F1={f1:.3f}")
    return {"model": model, "pred": pred, "acc": acc, "f1": f1,
            "prec": prec, "rec": rec}

cls_models = {
    "Random Forest"   : RandomForestClassifier(n_estimators=200, random_state=SEED, n_jobs=-1),
    "XGBoost"         : XGBClassifier(n_estimators=200, random_state=SEED,
                                       eval_metric="mlogloss", verbosity=0, use_label_encoder=False),
    "SVM (RBF)"       : SVC(kernel="rbf", C=10, gamma="scale", random_state=SEED),
    "MLP"             : MLPClassifier(hidden_layer_sizes=(256,128,64), max_iter=300,
                                      random_state=SEED, early_stopping=True),
}

cls_results = {}
print("── Classification Results ──────────────────────────────────")
for name, mdl in cls_models.items():
    cls_results[name] = eval_cls(name, mdl, X_tr_sc, y_cls_tr, X_te_sc, y_cls_te)

# Best model
best_cls_name = max(cls_results, key=lambda k: cls_results[k]["f1"])
best_cls = cls_results[best_cls_name]["model"]
print(f"\n✅ Best classifier: {best_cls_name}  (F1={cls_results[best_cls_name]['f1']:.3f})")
joblib.dump(best_cls, "best_classifier.pkl")
print("Saved → best_classifier.pkl")


In [ ]:
# Confusion matrix for best classifier
pred_best = cls_results[best_cls_name]["pred"]
cm = confusion_matrix(y_cls_te, pred_best)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=[f"MRC-{i}" for i in range(6)],
            yticklabels=[f"MRC-{i}" for i in range(6)], ax=ax)
ax.set_title(f"Confusion Matrix — {best_cls_name}", fontweight="bold")
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig("confusion_matrix.png", dpi=150)
plt.show()
print(classification_report(y_cls_te, pred_best, target_names=[f"MRC-{i}" for i in range(6)]))


## 7B — Regression: Predict Knee Angle

In [ ]:
def eval_reg(name, model, Xtr, ytr, Xte, yte):
    model.fit(Xtr, ytr)
    pred = model.predict(Xte)
    rmse = np.sqrt(mean_squared_error(yte, pred))
    mae  = mean_absolute_error(yte, pred)
    r2   = r2_score(yte, pred)
    print(f"{name:<25} RMSE={rmse:.3f}°  MAE={mae:.3f}°  R²={r2:.3f}")
    return {"model": model, "pred": pred, "rmse": rmse, "mae": mae, "r2": r2}

reg_models = {
    "Random Forest"   : RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
    "XGBoost"         : XGBRegressor(n_estimators=200, random_state=SEED, verbosity=0),
    "SVR (RBF)"       : SVR(kernel="rbf", C=10, gamma="scale"),
    "MLP"             : MLPRegressor(hidden_layer_sizes=(256,128,64), max_iter=300,
                                     random_state=SEED, early_stopping=True),
}

reg_results = {}
print("── Regression Results ──────────────────────────────────────")
for name, mdl in reg_models.items():
    reg_results[name] = eval_reg(name, mdl, X_tr_sc, y_sc_tr, X_te_sc, y_sc_te)

best_reg_name = min(reg_results, key=lambda k: reg_results[k]["rmse"])
best_reg = reg_results[best_reg_name]["model"]
print(f"\n✅ Best regressor: {best_reg_name}  (RMSE={reg_results[best_reg_name]['rmse']:.3f}°)")
joblib.dump(best_reg, "best_regressor.pkl")
print("Saved → best_regressor.pkl")


In [ ]:
# Prediction vs ground-truth plot
pred_reg = reg_results[best_reg_name]["pred"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(y_sc_te[:200], label="True angle", color="#2979FF", linewidth=1.5)
ax.plot(pred_reg[:200], label="Predicted",  color="#FF6D00", linewidth=1.5, alpha=0.85)
ax.set_xlabel("Window index"); ax.set_ylabel("Knee angle (°)")
ax.set_title(f"Knee Angle Prediction — {best_reg_name}", fontweight="bold")
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("regression_pred.png", dpi=150)
plt.show()


## 7C — Deep Learning: LSTM for Real-Time Movement Intention

Architecture optimised for embedded deployment:
- Lightweight two-layer LSTM with dropout
- Input: `(WINDOW_EMG=200, 4 channels)`
- Two heads: classification (MRC) + regression (knee angle)


In [ ]:
try:
    import tensorflow as tf
    from tensorflow.keras import layers, Model, callbacks

    tf.random.set_seed(SEED)

    # ── Shared LSTM encoder ───────────────────────────────────────────────────
    inp = layers.Input(shape=(WINDOW_EMG, len(CHANNELS)), name="emg_input")
    x   = layers.LSTM(64, return_sequences=True, name="lstm1")(inp)
    x   = layers.Dropout(0.3)(x)
    x   = layers.LSTM(32, return_sequences=False, name="lstm2")(x)
    x   = layers.Dropout(0.2)(x)
    enc = layers.Dense(64, activation="relu", name="encoder")(x)

    # ── Classification head (MRC 0-5) ─────────────────────────────────────────
    cls_head = layers.Dense(32, activation="relu")(enc)
    cls_out  = layers.Dense(6, activation="softmax", name="mrc_output")(cls_head)

    # ── Regression head (knee angle) ──────────────────────────────────────────
    reg_head = layers.Dense(32, activation="relu")(enc)
    reg_out  = layers.Dense(1, activation="linear", name="angle_output")(reg_head)

    lstm_model = Model(inputs=inp, outputs=[cls_out, reg_out], name="ExoskeletonLSTM")
    lstm_model.summary()

    # ── Compile ───────────────────────────────────────────────────────────────
    lstm_model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-3),
        loss={"mrc_output": "sparse_categorical_crossentropy",
              "angle_output": "mse"},
        loss_weights={"mrc_output": 1.0, "angle_output": 0.5},
        metrics={"mrc_output": "accuracy", "angle_output": "mae"}
    )

    # ── Callbacks ─────────────────────────────────────────────────────────────
    cb = [
        callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss"),
        callbacks.ReduceLROnPlateau(patience=5, factor=0.5, min_lr=1e-5),
    ]

    history = lstm_model.fit(
        X_seq_tr, {"mrc_output": y_cls_tr, "angle_output": y_sc_tr},
        validation_data=(X_seq_te, {"mrc_output": y_cls_te, "angle_output": y_sc_te}),
        epochs=60, batch_size=64,
        callbacks=cb, verbose=1
    )

    # ── Evaluate ──────────────────────────────────────────────────────────────
    cls_prob, angle_pred = lstm_model.predict(X_seq_te, verbose=0)
    cls_pred_lstm = np.argmax(cls_prob, axis=1)

    lstm_f1   = f1_score(y_cls_te, cls_pred_lstm, average="weighted", zero_division=0)
    lstm_rmse = np.sqrt(mean_squared_error(y_sc_te, angle_pred.flatten()))
    lstm_r2   = r2_score(y_sc_te, angle_pred.flatten())

    print(f"\nLSTM — Classification F1 : {lstm_f1:.3f}")
    print(f"LSTM — Regression  RMSE : {lstm_rmse:.3f}°")
    print(f"LSTM — Regression  R²   : {lstm_r2:.3f}")

    # ── Save in SavedModel + H5 formats ───────────────────────────────────────
    lstm_model.save("lstm_exoskeleton.h5")
    print("Saved → lstm_exoskeleton.h5")
    LSTM_AVAILABLE = True

except ImportError:
    print("TensorFlow not available. Skipping LSTM cell.")
    LSTM_AVAILABLE = False


In [ ]:
if LSTM_AVAILABLE:
    # Training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    axes[0].plot(history.history["mrc_output_accuracy"],    label="train")
    axes[0].plot(history.history["val_mrc_output_accuracy"],label="val")
    axes[0].set_title("LSTM — MRC Classification Accuracy")
    axes[0].set_xlabel("Epoch"); axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history.history["angle_output_mae"],     label="train")
    axes[1].plot(history.history["val_angle_output_mae"], label="val")
    axes[1].set_title("LSTM — Knee Angle MAE (°)")
    axes[1].set_xlabel("Epoch"); axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("lstm_training.png", dpi=150)
    plt.show()


## 8 — Model Comparison Summary

In [ ]:
print("\n══ CLASSIFICATION RESULTS ═════════════════════════════")
print(f"{'Model':<25} {'Acc':>6} {'Prec':>6} {'Recall':>7} {'F1':>6}")
print("-" * 55)
for name, r in cls_results.items():
    flag = " ✅" if name == best_cls_name else ""
    print(f"{name:<25} {r['acc']:>6.3f} {r['prec']:>6.3f} {r['rec']:>7.3f} {r['f1']:>6.3f}{flag}")

if LSTM_AVAILABLE:
    print(f"{'LSTM':<25} {'—':>6} {'—':>6} {'—':>7} {lstm_f1:>6.3f}")

print("\n══ REGRESSION RESULTS ══════════════════════════════════")
print(f"{'Model':<25} {'RMSE(°)':>8} {'MAE(°)':>7} {'R²':>6}")
print("-" * 50)
for name, r in reg_results.items():
    flag = " ✅" if name == best_reg_name else ""
    print(f"{name:<25} {r['rmse']:>8.3f} {r['mae']:>7.3f} {r['r2']:>6.3f}{flag}")

if LSTM_AVAILABLE:
    print(f"{'LSTM':<25} {lstm_rmse:>8.3f} {'—':>7} {lstm_r2:>6.3f}")


## 9 — ESP32 + NEMA17 Deployment Plan

### Architecture Overview

```
[EMG Electrodes] ──→ [ADS1299 / AD8232]
[IMU / Encoder]  ──→ [MPU6050 / AS5600]
                            │
                     [ESP32 (240 MHz)]
                            │
           ┌────────────────┼────────────────┐
    [Preprocess]      [Inference]      [Motor Control]
  HP→Rect→LP→Norm   TFLite Model     AccelStepper / RTOS
           └────────────────┼────────────────┘
                     [NEMA17 via A4988]
```

### Model Conversion for ESP32

1. **Quantise to INT8** — reduces model from ~400 KB to ~100 KB (fits in PSRAM):
   ```python
   converter = tf.lite.TFLiteConverter.from_keras_model(lstm_model)
   converter.optimizations = [tf.lite.Optimize.DEFAULT]
   tflite_model = converter.convert()
   open("model.tflite", "wb").write(tflite_model)
   ```

2. **Convert to C array** for embedding in firmware:
   ```bash
   xxd -i model.tflite > model_data.cc
   ```

3. **Include TFLite Micro** library in PlatformIO:
   ```ini
   lib_deps = tensorflow/TensorFlow Lite Micro
   ```

### Real-Time Inference Loop (pseudo-code)

```cpp
// ISR: 1 kHz ADC interrupt fills circular buffer
// Main loop: every 100ms (200 samples collected)
void loop() {
  if (window_ready) {
    preprocess_window(emg_buffer, norm_buffer);   // HP→Rect→LP→ZScore
    run_tflite_inference(norm_buffer, &mrc, &angle);
    int32_t steps = angle_to_steps(angle, GEAR_RATIO);
    stepper.moveTo(steps);
    stepper.run();
  }
}
```

### NEMA17 + A4988 Mapping

| MRC Grade | Target Assistance | Motor Torque % |
|-----------|------------------|----------------|
| 0–1       | Full assist       | 100 %          |
| 2–3       | Partial assist    | 60 %           |
| 4         | Light assist      | 30 %           |
| 5         | Resistive mode    | 0 / negative   |

### Latency Budget (ESP32-S3 @ 240 MHz)

| Stage | Time |
|-------|------|
| ADC sampling (200 samples) | 200 ms |
| Preprocessing (HP+LP+norm) | ~2 ms |
| TFLite INT8 LSTM inference | ~8 ms |
| Motor command + PWM | ~1 ms |
| **Total latency** | **~211 ms** |

> Target for exoskeleton control: < 250 ms ✔


In [ ]:
# TFLite export
if LSTM_AVAILABLE:
    try:
        import tensorflow as tf
        converter = tf.lite.TFLiteConverter.from_keras_model(lstm_model)
        converter.optimizations = [tf.lite.Optimize.DEFAULT]
        tflite_model = converter.convert()
        with open("lstm_exoskeleton_int8.tflite", "wb") as f:
            f.write(tflite_model)
        size_kb = len(tflite_model) / 1024
        print(f"TFLite INT8 model saved → lstm_exoskeleton_int8.tflite  ({size_kb:.1f} KB)")
    except Exception as e:
        print(f"TFLite export: {e}")

print("\n── Saved Models ──────────────────────────────────────────")
import glob
for f in sorted(glob.glob("*.pkl") + glob.glob("*.h5") + glob.glob("*.tflite")):
    size = os.path.getsize(f) / 1024
    print(f"  {f:<40} {size:>8.1f} KB")
